# 07 深度排序网络 (RankNet) 股票选择策略

## 论文参考

- **作者**: Yan Li
- **年份**: 2021
- **标题**: *Stock Portfolio Selection with Deep RankNet*

### 摘要

传统的股票预测方法侧重于预测绝对收益率，而排序学习(Learning-to-Rank)方法关注股票之间的
相对排名关系。本文提出使用深度RankNet进行股票组合选择：通过共享权重的MLP比较每对股票的
因子特征，使用pairwise cross-entropy损失训练网络判断哪只股票将跑赢另一只。测试阶段对
所有股票打分排序，选择Top-N构建投资组合。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### RankNet 排序学习

给定股票 $i$ 和 $j$ 的因子特征 $\mathbf{x}_i$ 和 $\mathbf{x}_j$，共享权重网络 $f_{\theta}$ 分别计算分数：

$$s_i = f_{\theta}(\mathbf{x}_i), \quad s_j = f_{\theta}(\mathbf{x}_j)$$

### Pairwise 概率建模

股票 $i$ 排名高于 $j$ 的预测概率：

$$P_{ij} = \sigma(s_i - s_j) = \frac{1}{1 + e^{-(s_i - s_j)}}$$

### 真实标签

根据未来收益率确定真实排序关系：

$$\bar{P}_{ij} = \begin{cases} 1 & \text{if } r_i > r_j \\ 0 & \text{if } r_i < r_j \\ 0.5 & \text{if } r_i = r_j \end{cases}$$

### Pairwise Cross-Entropy 损失

$$\mathcal{L} = -\bar{P}_{ij} \log P_{ij} - (1 - \bar{P}_{ij}) \log(1 - P_{ij})$$

### 组合构建

1. 对所有股票使用 $f_{\theta}$ 打分
2. 按分数从高到低排序
3. 选择Top-N股票等权配置

In [ ]:
# ============================================================
# Cell 3: 动画 - Pairwise 比较流程可视化
# ============================================================
import numpy as np
import plotly.graph_objects as go

def animate_ranknet_pairwise():
    """动画展示RankNet的成对比较过程"""
    np.random.seed(42)
    n_stocks = 8
    stock_names = [f'Stock {chr(65+i)}' for i in range(n_stocks)]
    scores = np.random.randn(n_stocks) * 0.5 + 0.5
    true_returns = np.random.randn(n_stocks) * 0.03

    # 生成一些典型比较对
    pairs = [(0,1),(2,3),(4,5),(6,7),(0,2),(1,3),(4,6),(5,7),(0,4),(1,5),(2,6),(3,7)]
    frames = []

    for step in range(len(pairs) + 1):
        shapes = []
        annotations = []

        # 左侧: 股票因子列表
        stock_x = []
        stock_y = []
        stock_colors = []
        stock_texts = []

        for i in range(n_stocks):
            stock_x.append(1)
            stock_y.append(n_stocks - i)
            stock_colors.append('#2196F3')
            stock_texts.append(f'{stock_names[i]}<br>score={scores[i]:.2f}')

        data = [
            go.Scatter(x=stock_x, y=stock_y, mode='markers+text',
                       marker=dict(size=30, color=stock_colors, opacity=0.8),
                       text=[stock_names[i] for i in range(n_stocks)],
                       textposition='middle center',
                       textfont=dict(size=9, color='white'),
                       hovertext=stock_texts, hoverinfo='text',
                       showlegend=False),
        ]

        # 中间: 共享网络
        data.append(go.Scatter(
            x=[4], y=[n_stocks/2 + 0.5], mode='markers+text',
            marker=dict(size=50, color='#FF9800', symbol='square', opacity=0.8),
            text=['Shared<br>MLP'], textposition='middle center',
            textfont=dict(size=10, color='white'),
            showlegend=False, hoverinfo='none'
        ))

        # 右侧: 比较结果
        if step > 0:
            compare_y = []
            compare_texts = []
            compare_colors = []
            for p_idx in range(min(step, len(pairs))):
                i, j = pairs[p_idx]
                prob = 1.0 / (1.0 + np.exp(-(scores[i] - scores[j])))
                winner = stock_names[i] if prob > 0.5 else stock_names[j]
                compare_y.append(n_stocks - p_idx * 0.7)
                compare_texts.append(f'{stock_names[i]} vs {stock_names[j]}: {winner} wins ({prob:.0%})')
                compare_colors.append('#4CAF50' if prob > 0.5 else '#F44336')

            data.append(go.Scatter(
                x=[7] * len(compare_y), y=compare_y, mode='text',
                text=compare_texts,
                textposition='middle right',
                textfont=dict(size=9),
                showlegend=False, hoverinfo='none'
            ))

            # 高亮当前比较对
            if step <= len(pairs):
                ci, cj = pairs[step - 1]
                for idx in [ci, cj]:
                    data.append(go.Scatter(
                        x=[1], y=[n_stocks - idx], mode='markers',
                        marker=dict(size=35, color='rgba(0,0,0,0)',
                                    line=dict(color='#FF9800', width=3)),
                        showlegend=False, hoverinfo='none'
                    ))

        frames.append(go.Frame(
            data=data,
            name=f'比较 {step}/{len(pairs)}' if step > 0 else '初始状态'
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)
    fig.update_layout(
        title=dict(text='RankNet Pairwise 比较动画'),
        xaxis=dict(visible=False, range=[0, 12]),
        yaxis=dict(visible=False, range=[-1, n_stocks + 1]),
        height=550, width=900,
        plot_bgcolor='white',
        updatemenus=[dict(type='buttons', x=0.1, y=1.12, buttons=[
            dict(label='\u25b6 播放', method='animate',
                 args=[None, {'frame': {'duration': 800}, 'transition': {'duration': 300}}]),
            dict(label='\u23f8 暂停', method='animate',
                 args=[[None], {'frame': {'duration': 0}, 'mode': 'immediate'}]),
        ])],
        sliders=[dict(
            steps=[dict(args=[[f.name], {'frame': {'duration': 0}, 'mode': 'immediate'}],
                        label=f.name, method='animate') for f in frames],
            active=0, x=0.05, len=0.9,
        )],
    )
    return fig

fig = animate_ranknet_pairwise()
fig.show()

In [ ]:
# ============================================================
# Cell 4: 导入与环境设置
# ============================================================
import sys, os, warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import StandardScaler
from itertools import combinations

from shared.data_fetcher import get_stock_daily, get_multiple_stocks_daily, get_csi300_constituents
from shared.backtest_engine import Backtester
from shared.visualization import plot_equity_curve, plot_metrics_table, set_chinese_font
from shared.factors import momentum, volatility, rsi, macd, bollinger_bands, price_to_ma
from shared.metrics import summary_table

set_chinese_font()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# ============================================================
# Cell 5: 数据获取 - 20只股票
# ============================================================
# 使用20只代表性A股
STOCK_POOL = [
    '600519', '000858', '601318', '600036', '601166',
    '600276', '601398', '600030', '000333', '002415',
    '600900', '601888', '600809', '000568', '002304',
    '601012', '600031', '603259', '600585', '000661',
]

stock_data = get_multiple_stocks_daily(
    STOCK_POOL, start_date='20200601', end_date='20241231'
)
print(f'成功获取 {len(stock_data)} 只股票数据')
for sym, df in list(stock_data.items())[:3]:
    print(f'  {sym}: {len(df)} 行, {df.index[0].date()} ~ {df.index[-1].date()}')

In [ ]:
# ============================================================
# Cell 6: 特征工程 - 构建每只股票每天的因子向量
# ============================================================
def build_stock_features(df):
    """为单只股票构建因子特征"""
    feat = pd.DataFrame(index=df.index)
    mom = momentum(df['close'], periods=[5, 10, 20])
    vol = volatility(df['close'], windows=[5, 10, 20])
    feat = pd.concat([feat, mom, vol], axis=1)
    feat['rsi_14'] = rsi(df['close'], 14)
    macd_df = macd(df['close'])
    feat['macd_hist'] = macd_df['histogram']
    bb = bollinger_bands(df['close'])
    feat['bb_pctb'] = bb['bb_pctb']
    feat['bb_width'] = bb['bb_width']
    ptm = price_to_ma(df['close'], windows=[5, 20])
    feat = pd.concat([feat, ptm], axis=1)
    # 未来5日收益 (用于训练标签)
    feat['fwd_ret_5d'] = df['close'].pct_change(5).shift(-5)
    return feat

# 构建所有股票因子
all_features = {}
for sym, df in stock_data.items():
    feat = build_stock_features(df)
    feat.dropna(inplace=True)
    if len(feat) > 100:
        all_features[sym] = feat

# 获取公共日期
common_dates = None
for sym, feat in all_features.items():
    if common_dates is None:
        common_dates = set(feat.index)
    else:
        common_dates &= set(feat.index)
common_dates = sorted(common_dates)

feature_cols = [c for c in list(all_features.values())[0].columns if c != 'fwd_ret_5d']
n_features = len(feature_cols)
print(f'股票数: {len(all_features)}, 公共日期: {len(common_dates)}, 特征维度: {n_features}')

In [ ]:
# ============================================================
# Cell 7: PyTorch RankNet 模型与训练
# ============================================================

class RankNet(nn.Module):
    """共享权重MLP用于排序打分"""
    def __init__(self, input_dim):
        super().__init__()
        self.scorer = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.scorer(x).squeeze(-1)

    def predict_pair(self, x_i, x_j):
        """预测 i 排名高于 j 的概率"""
        s_i = self.forward(x_i)
        s_j = self.forward(x_j)
        return torch.sigmoid(s_i - s_j)


class PairwiseDataset(Dataset):
    """生成Pairwise训练数据"""
    def __init__(self, features_dict, dates, feature_cols, max_pairs_per_date=50):
        self.pairs_x_i = []
        self.pairs_x_j = []
        self.labels = []
        symbols = list(features_dict.keys())

        for date in dates:
            # 获取当日所有股票的特征和未来收益
            day_data = []
            for sym in symbols:
                if date in features_dict[sym].index:
                    row = features_dict[sym].loc[date]
                    if not row[feature_cols].isna().any() and not np.isnan(row['fwd_ret_5d']):
                        day_data.append((sym, row[feature_cols].values, row['fwd_ret_5d']))

            if len(day_data) < 2:
                continue

            # 随机采样配对
            all_pairs = list(combinations(range(len(day_data)), 2))
            np.random.shuffle(all_pairs)
            selected = all_pairs[:max_pairs_per_date]

            for i, j in selected:
                self.pairs_x_i.append(day_data[i][1].astype(np.float32))
                self.pairs_x_j.append(day_data[j][1].astype(np.float32))
                # 标签: i的未来收益 > j的未来收益 → 1
                self.labels.append(1.0 if day_data[i][2] > day_data[j][2] else 0.0)

        self.pairs_x_i = np.array(self.pairs_x_i)
        self.pairs_x_j = np.array(self.pairs_x_j)
        self.labels = np.array(self.labels, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (torch.FloatTensor(self.pairs_x_i[idx]),
                torch.FloatTensor(self.pairs_x_j[idx]),
                torch.FloatTensor([self.labels[idx]]))


# --- 划分训练/测试集 ---
TRAIN_END = pd.Timestamp('2023-12-31')
TEST_START = pd.Timestamp('2024-01-01')

train_dates = [d for d in common_dates if d <= TRAIN_END]
test_dates = [d for d in common_dates if d >= TEST_START]

# 标准化
scaler = StandardScaler()
all_train_values = []
for sym, feat in all_features.items():
    mask = feat.index <= TRAIN_END
    all_train_values.append(feat.loc[mask, feature_cols].values)
all_train_values = np.vstack(all_train_values)
scaler.fit(all_train_values)

# 标准化所有特征
all_features_scaled = {}
for sym, feat in all_features.items():
    scaled = feat.copy()
    scaled[feature_cols] = scaler.transform(feat[feature_cols].values)
    all_features_scaled[sym] = scaled

# 采样训练日期 (减少计算量)
train_sample_dates = train_dates[::5]  # 每5天取一个
print(f'训练配对天数: {len(train_sample_dates)}, 测试天数: {len(test_dates)}')

# 构建数据集
train_dataset = PairwiseDataset(all_features_scaled, train_sample_dates,
                                 feature_cols, max_pairs_per_date=40)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
print(f'训练配对数: {len(train_dataset)}')

# --- 训练 ---
model = RankNet(input_dim=n_features).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)

EPOCHS = 40
train_losses = []

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    n_samples = 0
    for x_i, x_j, label in train_loader:
        x_i, x_j, label = x_i.to(device), x_j.to(device), label.squeeze().to(device)
        optimizer.zero_grad()
        prob = model.predict_pair(x_i, x_j)
        loss = criterion(prob, label)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(label)
        n_samples += len(label)
    scheduler.step()

    avg_loss = epoch_loss / max(n_samples, 1)
    train_losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}')

print('训练完成!')

In [ ]:
# ============================================================
# Cell 8: 回测 - 月度调仓选股
# ============================================================
# 每月初对所有股票打分，选Top-5等权持有
TOP_N = 5
symbols = list(all_features_scaled.keys())

# 获取测试期间每月第一个交易日
test_df_dates = pd.DatetimeIndex(test_dates)
monthly_rebalance = test_df_dates.to_series().groupby(test_df_dates.to_period('M')).first()

selections = {}  # {date: [sym1, sym2, ...]}
model.eval()

for rebal_date in monthly_rebalance.values:
    scores = {}
    with torch.no_grad():
        for sym in symbols:
            if rebal_date in all_features_scaled[sym].index:
                x = all_features_scaled[sym].loc[rebal_date, feature_cols].values
                if not np.isnan(x).any():
                    x_t = torch.FloatTensor(x).unsqueeze(0).to(device)
                    score = model(x_t).item()
                    scores[sym] = score

    if len(scores) >= TOP_N:
        top_stocks = sorted(scores, key=scores.get, reverse=True)[:TOP_N]
        selections[rebal_date] = top_stocks
        print(f'{rebal_date.date()}: Top-{TOP_N} = {top_stocks}')

# 构建价格字典
all_prices = {sym: stock_data[sym]['close'] for sym in symbols if sym in stock_data}

# 组合回测 - 简化版
from shared.backtest_engine import MultiStockBacktester
mbt = MultiStockBacktester(initial_capital=1_000_000, rebalance_freq='M')
result = mbt.run(all_prices, selections)

print('\n回测绩效指标:')
for k, v in result['metrics'].items():
    print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 9: 可视化
# ============================================================
import matplotlib.pyplot as plt

# 1. 训练损失
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_losses, color='#2196F3', linewidth=1.5)
ax.set_title('RankNet 训练损失 (Pairwise BCE)', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2. 收益曲线
equity = result['equity_curve']
test_mask = equity.index >= TEST_START
test_equity = equity.loc[test_mask]

if len(test_equity) > 0:
    plot_equity_curve(test_equity, title='RankNet 组合选股策略 - 收益曲线')

# 3. 绩效表格
plot_metrics_table(result['metrics'], title='RankNet (Li 2021) - 绩效指标')

# 4. 选股频率统计
from collections import Counter
all_selected = [s for stocks in selections.values() for s in stocks]
freq = Counter(all_selected)
freq_sorted = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(list(freq_sorted.keys())[::-1], list(freq_sorted.values())[::-1],
        color='#2196F3', alpha=0.8)
ax.set_title('RankNet 股票入选频率', fontsize=14, fontweight='bold')
ax.set_xlabel('入选次数')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 结果分析

### 模型特点

- **排序学习 vs 回归**: RankNet关注股票之间的相对排名而非绝对收益预测，更适合选股任务
- **共享权重**: 同一个MLP为所有股票打分，保证了可比性
- **Pairwise训练**: 通过配对比较构建训练样本，数据利用效率高

### 策略逻辑

1. 从20只候选股票中每月选出得分最高的5只
2. 等权配置，月度调仓
3. 核心假设：排名高的股票未来表现更好

### 局限性

- 股票池较小（20只），排序信息有限
- 未考虑行业中性化和市值风格暴露
- Pairwise样本数量随股票数二次增长，大规模应用需采样优化